In [2]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from typing import Optional
from tenacity import retry, stop_after_attempt, wait_exponential

ROOT = Path(r"C:\Users\MAITHILI\Care_Companion")

os.environ["GROQ_API_KEY"] = "********"
os.environ["USDA_API_KEY"] = "**********"

print("Evaluation notebook")
print(f"ROOT: {ROOT.absolute()}")

Evaluation notebook
ROOT: C:\Users\MAITHILI\Care_Companion


In [3]:
from sqlalchemy import create_engine, Column, String, Integer, Float, Text
from sqlalchemy.orm import declarative_base, Session
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict
import chromadb, re

Base    = declarative_base()
DB_PATH = ROOT / "memory" / "sqlite" / "carecompanion.db"

class Patient(Base):
    __tablename__    = "patients"
    patient_id       = Column(String,  primary_key=True)
    name             = Column(String)
    age              = Column(Integer)
    created_at       = Column(String)
    last_seen        = Column(String)
    conditions       = Column(Text)
    medications      = Column(Text)
    allergies        = Column(Text)
    recent_glucose   = Column(Float)
    recent_a1c       = Column(Float)
    recent_bmi       = Column(Float)
    diet_preferences = Column(Text)
    foods_to_avoid   = Column(Text)
    activity_level   = Column(String)
    budget_monthly   = Column(Float)
    insurance        = Column(String)
    session_count    = Column(Integer, default=0)
    agent_notes      = Column(Text)
    last_fda_check   = Column(String, nullable=True)

engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
Base.metadata.create_all(engine)

def load_patient(patient_id):
    with Session(engine) as s:
        row = s.get(Patient, patient_id)
        if row is None:
            return None
        return {
            "patient_id":       row.patient_id,
            "name":             row.name,
            "age":              row.age,
            "conditions":       json.loads(row.conditions),
            "medications":      json.loads(row.medications),
            "allergies":        json.loads(row.allergies),
            "recent_glucose":   row.recent_glucose,
            "recent_a1c":       row.recent_a1c,
            "recent_bmi":       row.recent_bmi,
            "diet_preferences": json.loads(row.diet_preferences),
            "foods_to_avoid":   json.loads(row.foods_to_avoid),
            "activity_level":   row.activity_level,
            "budget_monthly":   row.budget_monthly,
            "insurance":        row.insurance,
            "session_count":    row.session_count,
            "agent_notes":      json.loads(row.agent_notes),
            "last_fda_check":   row.last_fda_check,
            "created_at":       row.created_at,
            "last_seen":        row.last_seen,
        }

def update_patient_field(patient_id, field, value):
    with Session(engine) as s:
        row = s.get(Patient, patient_id)
        if row:
            setattr(row, field,
                    json.dumps(value) if isinstance(value, (list,dict))
                    else value)
            row.last_seen = datetime.now().isoformat()
            s.commit()

def list_all_patients():
    with Session(engine) as s:
        return [{"patient_id": r.patient_id, "name": r.name,
                 "age": r.age, "session_count": r.session_count}
                for r in s.query(Patient).all()]

chroma_client = chromadb.PersistentClient(
    path=str(ROOT / "memory" / "chroma_db"))
conversations = chroma_client.get_or_create_collection(
    "conversations", metadata={"hnsw:space": "cosine"})

def save_message(patient_id, role, content, session_id):
    doc_id = f"{patient_id}_{session_id}_{datetime.now().timestamp()}"
    conversations.add(
        documents=[content],
        metadatas=[{"patient_id": patient_id, "role": role,
                    "session_id": session_id,
                    "timestamp":  datetime.now().isoformat()}],
        ids=[doc_id]
    )

def get_relevant_history(patient_id, query, n=3):
    count = conversations.count()
    if count == 0:
        return []
    results = conversations.query(
        query_texts=[query],
        n_results=min(n, count),
        where={"patient_id": patient_id}
    )
    if not results["documents"][0]:
        return []
    return [{"role": m["role"], "content": d}
            for d, m in zip(results["documents"][0],
                            results["metadatas"][0])]

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1,
               api_key=os.environ["GROQ_API_KEY"])

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=5))
def check_drug_interactions(medications):
    results = {}
    for med in medications:
        drug_name = med["name"] if isinstance(med, dict) else med
        try:
            resp = requests.get(
                "https://api.fda.gov/drug/label.json",
                params={"search": f'openfda.generic_name:"{drug_name.lower()}"',
                        "limit": 1}, timeout=10)
            if resp.status_code != 200:
                results[drug_name] = {"status": "not_found"}
                continue
            label = resp.json()["results"][0]
            results[drug_name] = {
                "status":            "found",
                "drug_interactions": label.get("drug_interactions",["None"])[0][:400],
                "warnings":          label.get("warnings",["None"])[0][:300],
            }
        except Exception as e:
            results[drug_name] = {"status": "error", "error": str(e)}
    return results

def get_medication_cost(drug_name):
    PRICES = {
        "metformin":        {"brand":"Glucophage","brand_price":45,"generic_price":4},
        "glipizide":        {"brand":"Glucotrol", "brand_price":38,"generic_price":9},
        "sitagliptin":      {"brand":"Januvia",   "brand_price":550,"generic_price":45},
        "lisinopril":       {"brand":"Zestril",   "brand_price":65,"generic_price":8},
        "insulin glargine": {"brand":"Lantus",    "brand_price":340,"generic_price":98},
    }
    info = PRICES.get(drug_name.lower())
    if info:
        savings = info["brand_price"] - info["generic_price"]
        return {"drug_name": drug_name, "generic_price": info["generic_price"],
                "brand_price": info["brand_price"], "monthly_savings": savings,
                "annual_savings": savings * 12}
    return {"drug_name": drug_name, "generic_price": None, "annual_savings": 0}

def lookup_nutrition(food_name):
    try:
        resp = requests.get(
            "https://api.nal.usda.gov/fdc/v1/foods/search",
            params={"query": food_name, "pageSize": 1,
                    "api_key": os.environ.get("USDA_API_KEY","DEMO_KEY")},
            timeout=15)
        foods = resp.json().get("foods", [])
        if not foods:
            return {"status": "not_found"}
        nraw = {n["nutrientName"]: n["value"]
                for n in foods[0].get("foodNutrients", [])}
        net_carbs = max(0, nraw.get("Carbohydrate, by difference",0)
                           - nraw.get("Fiber, total dietary",0))
        if net_carbs < 5:    flag = "green"
        elif net_carbs < 15: flag = "yellow"
        elif net_carbs < 30: flag = "orange"
        else:                flag = "red"
        return {"net_carbs_g": round(net_carbs,1), "flag": flag, "status": "found"}
    except Exception:
        return {"status": "error"}

def get_fda_alerts(medications):
    alerts = {}
    for med in medications:
        drug_name = med["name"] if isinstance(med, dict) else med
        drug_alerts = []
        try:
            resp = requests.get(
                "https://api.fda.gov/drug/enforcement.json",
                params={"search": f'product_description:"{drug_name}"',
                        "limit": 2}, timeout=10)
            if resp.status_code == 200:
                for r in resp.json().get("results",[]):
                    drug_alerts.append(r.get("reason_for_recall","")[:100])
        except Exception:
            pass
        alerts[drug_name] = {"alerts_found": len(drug_alerts),
                             "alerts": drug_alerts}
    return alerts

print("loading all components")
patients = list_all_patients()
print(f"Patients in DB: {len(patients)}")

loading all components
Patients in DB: 20


In [4]:
import json, numpy as np
from pathlib import Path
from datetime import datetime

ROOT = Path("carecompanion-ai").absolute()

# Load existing 20 patients
with open(ROOT / "evaluation" / "test_patients.json",
          "r", encoding="utf-8") as f:
    existing = json.load(f)

print(f"Current patients: {len(existing)}")

#expanded the dataset for evaluation purpose
ALL_MEDS = [

    [{"name":"Metformin",        "dose":"500mg",    "frequency":"twice daily"}],
    [{"name":"Metformin",        "dose":"1000mg",   "frequency":"twice daily"}],
    [{"name":"Glipizide",        "dose":"5mg",      "frequency":"once daily"}],
    [{"name":"Sitagliptin",      "dose":"100mg",    "frequency":"once daily"}],
    [{"name":"Insulin Glargine", "dose":"20 units", "frequency":"once daily"}],

    [{"name":"Metformin",  "dose":"1000mg","frequency":"twice daily"},
     {"name":"Glipizide",  "dose":"5mg",   "frequency":"once daily"}],
    [{"name":"Metformin",  "dose":"500mg", "frequency":"twice daily"},
     {"name":"Lisinopril", "dose":"10mg",  "frequency":"once daily"}],
    [{"name":"Metformin",  "dose":"1000mg","frequency":"twice daily"},
     {"name":"Sitagliptin","dose":"100mg", "frequency":"once daily"}],

    [{"name":"Metformin",        "dose":"1000mg",   "frequency":"twice daily"},
     {"name":"Glipizide",        "dose":"5mg",      "frequency":"once daily"},
     {"name":"Lisinopril",       "dose":"10mg",     "frequency":"once daily"}],
    [{"name":"Insulin Glargine", "dose":"20 units", "frequency":"once daily"},
     {"name":"Metformin",        "dose":"500mg",    "frequency":"twice daily"},
     {"name":"Lisinopril",       "dose":"10mg",     "frequency":"once daily"}],
]

# More names
NAMES = [
    "Aisha","Benjamin","Carmen","Derek","Elena","Farid","Gloria","Hassan",
    "Irene","Jerome","Kavya","Liam","Maya","Nathan","Olivia","Pedro",
    "Quinn","Rosa","Samuel","Tara","Uma","Victor","Wanda","Xavier",
    "Yara","Zoe","Arjun","Bella","Cyrus","Diana","Evan","Fatima",
    "George","Hana","Ivan","Julia","Kenji","Luna","Marco","Nina",
    "Oscar","Priya","Raj","Sofia","Tariq","Ursula","Vance","Wendy",
    "Xander","Yasmin","Zara","Ahmed","Bianca","Carlos","Dina","Eric"
]

INSURANCE  = ["none", "none", "medicaid", "medicaid", "private"]  # weighted realistic
ACTIVITY   = ["sedentary", "sedentary", "light", "moderate", "active"]  # weighted
DIETS      = [
    ["low-carb"], ["low-carb"],
    ["vegetarian"], ["mediterranean"],
    ["standard"], ["standard"],
    ["low-carb", "vegetarian"],
    ["mediterranean", "low-sodium"]
]
BUDGETS    = [50, 75, 75, 100, 100, 150, 200, 300]
CONDITIONS = [
    ["Type 2 Diabetes"],
    ["Type 2 Diabetes"],
    ["Type 2 Diabetes"],
    ["Type 2 Diabetes", "Hypertension"],
    ["Type 2 Diabetes", "Hypertension"],
    ["Type 2 Diabetes", "High Cholesterol"],
    ["Type 2 Diabetes", "Obesity"],
    ["Type 2 Diabetes", "Hypertension", "High Cholesterol"],
]

#loading Pima dataset for realistic clinical values
import pandas as pd
df = pd.read_csv(ROOT / "data" / "processed" / "diabetes_clean.csv")
diabetic_df = df[df["Outcome"] == 1].reset_index(drop=True)

np.random.seed(99)
new_patients = []

#generating 50 new patients
used_names = {p["name"] for p in existing}
available  = [n for n in NAMES if n not in used_names]

for i in range(50):
    row = diabetic_df.iloc[i % len(diabetic_df)]

    # Add realistic noise to clinical values
    glucose = float(row["Glucose"]) + np.random.normal(0, 15)
    glucose = max(70, min(350, glucose))
    bmi     = float(row["BMI"]) + np.random.normal(0, 2)
    bmi     = max(18, min(55, bmi))
    a1c     = round(glucose / 28.7 + 2.15 + np.random.normal(0, 0.3), 1)
    a1c     = max(5.5, min(14.0, a1c))
    age     = int(row["Age"]) + np.random.randint(-3, 4)
    age     = max(25, min(80, age))

    name = available[i % len(available)]

    p = {
        "patient_id":       f"p{len(existing)+i+1:03d}",
        "name":             name,
        "age":              age,
        "created_at":       datetime.now().isoformat(),
        "last_seen":        datetime.now().isoformat(),
        "conditions":       CONDITIONS[i % len(CONDITIONS)],
        "medications":      ALL_MEDS[i % len(ALL_MEDS)],
        "allergies":        [] if i % 5 != 0 else ["sulfa drugs"],
        "recent_glucose":   round(glucose, 1),
        "recent_a1c":       a1c,
        "recent_bmi":       round(bmi, 1),
        "diet_preferences": DIETS[i % len(DIETS)],
        "foods_to_avoid":   [],
        "activity_level":   ACTIVITY[i % len(ACTIVITY)],
        "budget_monthly":   float(np.random.choice(BUDGETS)),
        "insurance":        INSURANCE[i % len(INSURANCE)],
        "session_count":    0,
        "agent_notes":      [],
        "last_fda_check":   None,
    }
    new_patients.append(p)

all_patients_expanded = existing + new_patients

#saving expanded dataset
save_path = ROOT / "evaluation" / "test_patients_expanded.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(all_patients_expanded, f, indent=2)

print(f"Expanded dataset: {len(all_patients_expanded)} patients")
print(f"Saved to: {save_path}")
print()

all_meds_flat = []
for p in all_patients_expanded:
    all_meds_flat.extend([m["name"] for m in p["medications"]])

from collections import Counter
med_counts = Counter(all_meds_flat)
print("Medication distribution:")
for med, count in med_counts.most_common():
    bar = "█" * (count // 2)
    print(f"  {med:20s} {count:3d} {bar}")

print()
ins_counts = Counter(p["insurance"] for p in all_patients_expanded)
print("Insurance distribution:")
for ins, count in ins_counts.items():
    print(f"  {ins:12s}: {count} patients")

print()
budget_vals = [p["budget_monthly"] for p in all_patients_expanded]
print(f"Budget range: ${min(budget_vals):.0f} — ${max(budget_vals):.0f}/month")
print(f"Budget mean:  ${np.mean(budget_vals):.0f}/month")

Current patients: 20
Expanded dataset: 70 patients
Saved to: C:\Users\MAITHILI\Care_Companion\codes\carecompanion-ai\evaluation\test_patients_expanded.json

Medication distribution:
  Metformin             47 ███████████████████████
  Glipizide             19 █████████
  Lisinopril            19 █████████
  Insulin Glargine      14 ███████
  Sitagliptin           14 ███████

Insurance distribution:
  none        : 27 patients
  medicaid    : 27 patients
  private     : 16 patients

Budget range: $50 — $300/month
Budget mean:  $152/month


In [5]:
#loading expanded patients into SQLite
from sqlalchemy.orm import Session
import json

with open(ROOT / "evaluation" / "test_patients_expanded.json",
          "r", encoding="utf-8") as f:
    expanded = json.load(f)

def save_patient(patient):
    with Session(engine) as s:
        row = Patient(
            patient_id       = patient["patient_id"],
            name             = patient["name"],
            age              = patient["age"],
            created_at       = patient["created_at"],
            last_seen        = patient["last_seen"],
            conditions       = json.dumps(patient["conditions"]),
            medications      = json.dumps(patient["medications"]),
            allergies        = json.dumps(patient["allergies"]),
            recent_glucose   = patient["recent_glucose"],
            recent_a1c       = patient["recent_a1c"],
            recent_bmi       = patient["recent_bmi"],
            diet_preferences = json.dumps(patient["diet_preferences"]),
            foods_to_avoid   = json.dumps(patient["foods_to_avoid"]),
            activity_level   = patient["activity_level"],
            budget_monthly   = patient["budget_monthly"],
            insurance        = patient["insurance"],
            session_count    = patient["session_count"],
            agent_notes      = json.dumps(patient["agent_notes"]),
            last_fda_check   = patient["last_fda_check"],
        )
        s.merge(row)
        s.commit()

#loading new ones
existing_ids = {p["patient_id"] for p in existing}
new_only     = [p for p in expanded if p["patient_id"] not in existing_ids]

for p in new_only:
    save_patient(p)

print(f"Added {len(new_only)} new patients to database")
print(f"Total in DB: {len(expanded)}")

#recalculateing cost savings on full 70 patients
PRICES = {
    "metformin":        {"brand_price": 45,  "generic_price": 4},
    "glipizide":        {"brand_price": 38,  "generic_price": 9},
    "sitagliptin":      {"brand_price": 550, "generic_price": 45},
    "lisinopril":       {"brand_price": 65,  "generic_price": 8},
    "insulin glargine": {"brand_price": 340, "generic_price": 98},
}

savings_list = []
for p in expanded:
    total = 0
    for med in p["medications"]:
        info = PRICES.get(med["name"].lower(), {})
        if info:
            total += (info["brand_price"] - info["generic_price"]) * 12
    savings_list.append(total)

import numpy as np
print(f"\nCost savings across {len(expanded)} patients:")
print(f"  Average: ${np.mean(savings_list):,.0f}/year")
print(f"  Median:  ${np.median(savings_list):,.0f}/year")
print(f"  Min:     ${np.min(savings_list):,.0f}/year")
print(f"  Max:     ${np.max(savings_list):,.0f}/year")
print(f"  Patients with triple therapy (highest savings): "
      f"{sum(1 for p in expanded if len(p['medications'])==3)}")

Added 50 new patients to database
Total in DB: 70

Cost savings across 70 patients:
  Average: $2,403/year
  Median:  $1,176/year
  Min:     $348/year
  Max:     $6,552/year
  Patients with triple therapy (highest savings): 10


In [6]:
print("METRIC 1: Tool Call Success Rate")
print("Harder test: includes edge cases and less common drugs")
print("-" * 50)

from tenacity import retry, stop_after_attempt, wait_exponential

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=5))
def check_openfda(drug_name):
    r = requests.get(
        "https://api.fda.gov/drug/label.json",
        params={"search": f'openfda.generic_name:"{drug_name.lower()}"',
                "limit": 1}, timeout=10)
    return r.status_code == 200 and "results" in r.json()

def check_usda(food_name):
    r = requests.get(
        "https://api.nal.usda.gov/fdc/v1/foods/search",
        params={"query": food_name, "pageSize": 1,
                "api_key": os.environ.get("USDA_API_KEY", "DEMO_KEY")},
        timeout=15)
    return r.status_code == 200 and len(r.json().get("foods", [])) > 0

def check_rxnorm(drug_name):
    r = requests.get(
        "https://rxnav.nlm.nih.gov/REST/rxcui.json",
        params={"name": drug_name}, timeout=10)
    return bool(r.json().get("idGroup", {}).get("rxnormId"))

#harder test cases — includes edge cases
drug_tests = [

    ("Metformin",        "openfda", True),
    ("Lisinopril",       "openfda", True),
    ("Glipizide",        "openfda", True),
    ("Atorvastatin",     "openfda", True),
    ("Amlodipine",       "openfda", True),

    ("Empagliflozin",    "openfda", None),
    ("Dapagliflozin",    "openfda", None),
    ("Semaglutide",      "openfda", None),

    ("Metformiiin",      "openfda", False),
    ("Randomdrug123",    "openfda", False),
]

food_tests = [
    ("broccoli",    True),
    ("white rice",  True),
    ("quinoa",      True),
    ("chapati",     None),   
    ("idli",        None),   
    ("chicken breast", True),
    ("xyz_fake_food_123", False),
]

rxnorm_tests = [
    ("metformin",   True),
    ("lisinopril",  True),
    ("aspirin",     True),
    ("ibuprofen",   True),
    ("fake_drug",   False),
]

results = []

print("OpenFDA tests:")
for drug, source, expected in drug_tests:
    try:
        success = check_openfda(drug)
        results.append({"tool": "OpenFDA", "input": drug,
                        "success": success, "expected": expected})
        status = "OK  " if success else "FAIL"
        note   = "" if expected is None else ("yes" if success==expected else " unexpected")
        print(f"  [{status}] {drug:25s}{note}")
        time.sleep(0.3)
    except Exception as e:
        results.append({"tool": "OpenFDA", "input": drug,
                        "success": False, "expected": expected})
        print(f"  [ERR ] {drug:25s}: {e}")

print("\nUSDA FoodData tests:")
for food, expected in food_tests:
    try:
        success = check_usda(food)
        results.append({"tool": "USDA", "input": food,
                        "success": success, "expected": expected})
        status = "OK  " if success else "FAIL"
        note   = "" if expected is None else ("yes" if success==expected else " unexpected")
        print(f"  [{status}] {food:25s}{note}")
        time.sleep(0.5)
    except Exception as e:
        results.append({"tool": "USDA", "input": food,
                        "success": False, "expected": expected})
        print(f"  [ERR ] {food:25s}: {e}")

print("\nRxNorm tests:")
for drug, expected in rxnorm_tests:
    try:
        success = check_rxnorm(drug)
        results.append({"tool": "RxNorm", "input": drug,
                        "success": success, "expected": expected})
        status = "OK  " if success else "FAIL"
        print(f"  [{status}] {drug:25s}")
        time.sleep(0.3)
    except Exception as e:
        results.append({"tool": "RxNorm", "input": drug,
                        "success": False, "expected": expected})

# Only counting tests where we had a clear expected outcome
definitive = [r for r in results if r["expected"] is not None]
correct    = [r for r in definitive if r["success"] == r["expected"]]
graceful   = [r for r in results if r["success"]] 

success_rate  = len(graceful) / len(results) * 100
accuracy_rate = len(correct)  / len(definitive) * 100

print(f"\nAPI call success rate:    {success_rate:.1f}%  ({len(graceful)}/{len(results)} calls returned data)")
print(f"Expected outcome match:   {accuracy_rate:.1f}%  ({len(correct)}/{len(definitive)} matched expectation)")

METRIC_TOOL_SUCCESS = success_rate
METRIC_TOOL_ACCURACY = accuracy_rate

METRIC 1: Tool Call Success Rate
Harder test: includes edge cases and less common drugs
--------------------------------------------------
OpenFDA tests:
  [OK  ] Metformin                yes
  [OK  ] Lisinopril               yes
  [OK  ] Glipizide                yes
  [OK  ] Atorvastatin             yes
  [OK  ] Amlodipine               yes
  [OK  ] Empagliflozin            
  [OK  ] Dapagliflozin            
  [OK  ] Semaglutide              
  [FAIL] Metformiiin              yes
  [FAIL] Randomdrug123            yes

USDA FoodData tests:
  [OK  ] broccoli                 yes
  [OK  ] white rice               yes
  [OK  ] quinoa                   yes
  [OK  ] chapati                  
  [OK  ] idli                     
  [OK  ] chicken breast           yes
  [FAIL] xyz_fake_food_123        yes

RxNorm tests:
  [OK  ] metformin                
  [OK  ] lisinopril               
  [OK  ] aspirin                  
  [OK  ] ibuprofen                
  [FAIL] fake_drug                

AP

In [7]:
print("METRIC 2: Memory Retrieval Accuracy")
print("Realistic test: multiple patients, noisy data, abstract queries")
print("-" * 55)

#storing some conversations for multiple patients, so ChromaDB has to distinguish between them
test_conversations = [
    # p001 Sarah — talks about rice and cost
    ("p001", "user",      "s_test1", "I have been eating white rice every day"),
    ("p001", "assistant", "s_test1", "White rice spikes blood sugar significantly"),
    ("p001", "user",      "s_test1", "My metformin is getting very expensive"),
    ("p001", "assistant", "s_test1", "Generic metformin costs only $4 per month"),

    # p002 James — talks about exercise and oatmeal
    ("p002", "user",      "s_test2", "I started walking 30 minutes every morning"),
    ("p002", "assistant", "s_test2", "Exercise is excellent for blood sugar control"),
    ("p002", "user",      "s_test2", "I eat oatmeal for breakfast every day"),
    ("p002", "assistant", "s_test2", "Oatmeal has high net carbs, consider alternatives"),

    # p003 Maria — talks about insulin and doctor visit
    ("p003", "user",      "s_test3", "My doctor increased my insulin dose last week"),
    ("p003", "assistant", "s_test3", "Insulin adjustment is common as diabetes progresses"),
    ("p003", "user",      "s_test3", "I had a hypoglycemia episode yesterday"),
    ("p003", "assistant", "s_test3", "Low blood sugar episodes need immediate attention"),
]

#storing all these
for pid, role, sid, content in test_conversations:
    save_message(pid, role, content, sid)

print(f"Stored {len(test_conversations)} messages across 3 patients")
print()
import time
time.sleep(1)  


hard_tests = [
    {
        "description":  "Food recall — p001",
        "patient_id":   "p001",
        # Query uses 'carbohydrates' but stored content uses 'rice'
        "query":        "what carbohydrates did this patient consume",
        "expected_kw":  ["rice", "oatmeal", "carb"],
        "wrong_kw":     ["oatmeal", "walking", "insulin"], 
    },
    {
        "description":  "Cost recall — p001",
        "patient_id":   "p001",
        # Query uses 'financial' but stored uses 'expensive'
        "query":        "financial concerns about treatment",
        "expected_kw":  ["expensive", "cost", "metformin", "generic"],
        "wrong_kw":     ["insulin", "exercise", "hypoglycemia"],
    },
    {
        "description":  "Exercise recall — p002",
        "patient_id":   "p002",
        # Query uses 'physical activity' but stored uses 'walking'
        "query":        "what physical activity does this patient do",
        "expected_kw":  ["walking", "exercise", "morning"],
        "wrong_kw":     ["rice", "metformin", "insulin"],  
    },
    {
        "description":  "Breakfast recall — p002",
        "patient_id":   "p002",
        # Query uses 'morning meal' but stored uses 'breakfast/oatmeal'
        "query":        "what does the patient eat in the morning",
        "expected_kw":  ["oatmeal", "breakfast", "carb"],
        "wrong_kw":     ["rice", "insulin", "hypoglycemia"],
    },
    {
        "description":  "Medical event — p003",
        "patient_id":   "p003",
        # Query uses 'blood sugar emergency' but stored uses 'hypoglycemia'
        "query":        "any blood sugar emergency or crisis",
        "expected_kw":  ["hypoglycemia", "low blood sugar", "insulin"],
        "wrong_kw":     ["rice", "oatmeal", "walking", "metformin"],
    },
    {
        "description":  "Cross-patient isolation — p001",
        "patient_id":   "p001",
        # Makes sure p002 exercise content doesn't bleed into p001 results
        "query":        "exercise and physical activity routine",
        "expected_kw":  [],  # p001 never talked about exercise
        "wrong_kw":     ["walking", "oatmeal"],  
        "expect_empty": True
    },
]

recall_results = []

for test in hard_tests:
    history  = get_relevant_history(test["patient_id"], test["query"], n=3)
    all_text = " ".join(m["content"].lower() for m in history)

    if test.get("expect_empty"):
        # Success = wrong patient content NOT retrieved
        wrong_found = [w for w in test["wrong_kw"] if w in all_text]
        success     = len(wrong_found) == 0
        matched     = f"correctly excluded: {test['wrong_kw']}"
        if not success:
            matched = f"WRONGLY included: {wrong_found}"
    else:
        # Success = right content found AND wrong patient content excluded
        matched_kw  = [kw for kw in test["expected_kw"] if kw in all_text]
        wrong_found = [w for w in test["wrong_kw"] if w in all_text]
        success     = len(matched_kw) > 0 and len(wrong_found) == 0
        matched     = f"found={matched_kw} | wrong={wrong_found}"

    recall_results.append({
        "test":    test["description"],
        "success": success,
        "detail":  matched,
        "msgs":    len(history)
    })

    status = "PASS" if success else "FAIL"
    print(f"  [{status}] {test['description']:35s} {matched}")

accuracy = (sum(1 for r in recall_results if r["success"])
            / len(recall_results) * 100)

print()
print(f"Memory Retrieval Accuracy: {accuracy:.1f}%")
print()
print("What this tests:")
print("  - Semantic understanding (different words, same meaning)")
print("  - Patient isolation (p001 query doesn't return p002 memories)")
print("  - Abstract queries (no shared vocabulary with stored content)")

METRIC_MEMORY_ACCURACY = accuracy


METRIC 2: Memory Retrieval Accuracy
Realistic test: multiple patients, noisy data, abstract queries
-------------------------------------------------------
Stored 12 messages across 3 patients

  [FAIL] Food recall — p001                  found=['rice', 'oatmeal', 'carb'] | wrong=['oatmeal']
  [PASS] Cost recall — p001                  found=['expensive', 'cost', 'metformin', 'generic'] | wrong=[]
  [PASS] Exercise recall — p002              found=['walking', 'exercise', 'morning'] | wrong=[]
  [PASS] Breakfast recall — p002             found=['oatmeal', 'breakfast', 'carb'] | wrong=[]
  [PASS] Medical event — p003                found=['hypoglycemia', 'low blood sugar'] | wrong=[]
  [FAIL] Cross-patient isolation — p001      WRONGLY included: ['oatmeal']

Memory Retrieval Accuracy: 66.7%

What this tests:
  - Semantic understanding (different words, same meaning)
  - Patient isolation (p001 query doesn't return p002 memories)
  - Abstract queries (no shared vocabulary with stored cont

In [8]:
print("=" * 55)
print("METRIC 3: Cost Savings Identified Per Patient")
print("=" * 55)

with open(ROOT / "evaluation" / "test_patients.json",
          "r", encoding="utf-8") as f:
    test_patients = json.load(f)

savings_data = []

for patient in test_patients:
    patient_total_savings = 0
    for med in patient["medications"]:
        result = get_medication_cost(med["name"])
        annual = result.get("annual_savings", 0)
        patient_total_savings += annual

    savings_data.append({
        "patient_id":    patient["patient_id"],
        "name":          patient["name"],
        "insurance":     patient["insurance"],
        "annual_savings": patient_total_savings,
    })

df_savings = pd.DataFrame(savings_data)

print(f"{'Name':12s} {'Insurance':10s} {'Annual Savings':>15s}")
print("-" * 42)
for _, row in df_savings.iterrows():
    print(f"  {row['name']:12s} {row['insurance']:10s} "
          f"${row['annual_savings']:>12,.0f}")

avg_savings = df_savings["annual_savings"].mean()
max_savings = df_savings["annual_savings"].max()

print()
print(f"Average annual savings per patient: ${avg_savings:,.0f}")
print(f"Maximum annual savings identified:  ${max_savings:,.0f}")
METRIC_AVG_COST_SAVINGS = avg_savings

METRIC 3: Cost Savings Identified Per Patient
Name         Insurance   Annual Savings
------------------------------------------
  Sarah        none       $         492
  James        medicaid   $         840
  Maria        private    $       1,176
  David        none       $       2,904
  Linda        medicaid   $       6,060
  Carlos       private    $         492
  Emma         none       $         840
  Michael      medicaid   $       1,176
  Priya        private    $       2,904
  Robert       none       $       6,060
  Angela       medicaid   $         492
  Kevin        private    $         840
  Susan        none       $       1,176
  Ahmed        medicaid   $       2,904
  Nicole       private    $       6,060
  Thomas       none       $         492
  Grace        medicaid   $         840
  Daniel       private    $       1,176
  Patricia     none       $       2,904
  Marcus       medicaid   $       6,060

Average annual savings per patient: $2,294
Maximum annual savings iden

In [9]:
print("=" * 55)
print("METRIC 4: Nutrition Assessment Accuracy")
print("=" * 55)

#ground truth: known glycemic classifications
#source form American Diabetes Association guidelines
nutrition_ground_truth = [
    {"food": "broccoli",      "expected_flag": "green",  "net_carbs": 4.0},
    {"food": "white rice",    "expected_flag": "red",    "net_carbs": 28.0},
    {"food": "brown rice",    "expected_flag": "red",    "net_carbs": 22.0},
    {"food": "chicken",       "expected_flag": "green",  "net_carbs": 0.0},
    {"food": "apple",         "expected_flag": "yellow", "net_carbs": 14.0},
    {"food": "oatmeal",       "expected_flag": "red",    "net_carbs": 23.0},
    {"food": "cauliflower",   "expected_flag": "green",  "net_carbs": 3.0},
    {"food": "pasta",         "expected_flag": "red",    "net_carbs": 31.0},
    {"food": "egg",           "expected_flag": "green",  "net_carbs": 1.0},
    {"food": "orange juice",  "expected_flag": "red",    "net_carbs": 25.0},
]

nutrition_results = []

for item in nutrition_ground_truth:
    result = lookup_nutrition(item["food"])
    time.sleep(0.3)

    if result["status"] == "found":
        actual_flag = result["flag"]
        correct     = actual_flag == item["expected_flag"]

        #partial credit to adjacent flags (green/yellow or yellow/orange)
        flag_order  = ["green","yellow","orange","red"]
        expected_i  = flag_order.index(item["expected_flag"])
        actual_i    = flag_order.index(actual_flag)
        adjacent    = abs(expected_i - actual_i) <= 1

        nutrition_results.append({
            "food":     item["food"],
            "expected": item["expected_flag"],
            "actual":   actual_flag,
            "correct":  correct,
            "adjacent": adjacent,
        })

        status = "EXACT" if correct else ("CLOSE" if adjacent else "WRONG")
        print(f"  [{status:5s}] {item['food']:15s} "
              f"expected={item['expected_flag']:7s} "
              f"got={actual_flag}")
    else:
        print(f"  [ERROR] {item['food']:15s} — API returned no result")

exact_acc   = sum(1 for r in nutrition_results if r["correct"])     / len(nutrition_results) * 100
partial_acc = sum(1 for r in nutrition_results if r["adjacent"])    / len(nutrition_results) * 100

print()
print(f"Exact accuracy:   {exact_acc:.1f}%")
print(f"Adjacent accuracy (within 1 flag): {partial_acc:.1f}%")
METRIC_NUTRITION_ACCURACY = exact_acc

METRIC 4: Nutrition Assessment Accuracy
  [EXACT] broccoli        expected=green   got=green
  [EXACT] white rice      expected=red     got=red
  [EXACT] brown rice      expected=red     got=red
  [EXACT] chicken         expected=green   got=green
  [EXACT] apple           expected=yellow  got=yellow
  [EXACT] oatmeal         expected=red     got=red
  [WRONG] cauliflower     expected=green   got=orange
  [EXACT] pasta           expected=red     got=red
  [WRONG] egg             expected=green   got=red
  [WRONG] orange juice    expected=red     got=yellow

Exact accuracy:   70.0%
Adjacent accuracy (within 1 flag): 70.0%


In [10]:
print("=" * 55)
print("METRIC 5: FDA Alert Coverage")
print("=" * 55)

all_drugs = set()
for p in test_patients:
    for med in p["medications"]:
        all_drugs.add(med["name"])

print(f"Unique medications across all patients: {len(all_drugs)}")
print()

alert_results = {}
for drug in all_drugs:
    result = get_fda_alerts([{"name": drug}])
    found  = result[drug]["alerts_found"]
    alert_results[drug] = found
    status = f"{found} alert(s)" if found > 0 else "clean"
    print(f"  {drug:25s}: {status}")
    time.sleep(0.2)

coverage = sum(1 for v in alert_results.values() if v >= 0) / len(alert_results) * 100
flagged  = sum(1 for v in alert_results.values() if v > 0)

print()
print(f"Drugs successfully scanned: {coverage:.1f}%")
print(f"Drugs with active alerts:   {flagged}/{len(all_drugs)}")
METRIC_FDA_COVERAGE = coverage

METRIC 5: FDA Alert Coverage
Unique medications across all patients: 5

  Sitagliptin              : 2 alert(s)
  Glipizide                : 2 alert(s)
  Lisinopril               : 2 alert(s)
  Metformin                : 2 alert(s)
  Insulin Glargine         : 2 alert(s)

Drugs successfully scanned: 100.0%
Drugs with active alerts:   5/5


In [11]:
print("\nMETRIC 3: Full Agent Loop Latency")
print("Measuring complete pipeline: context + tools + LLM response")
print("-" * 50)

test_cases = [
    ("p001", "Is idli okay for breakfast?"),
    ("p001", "How much could I save on my medications?"),
    ("p002", "What foods should I avoid?"),
    ("p003", "My glucose was 140 today, is that okay?"),
    ("p001", "Tell me about the side effects of my medication"),
]

latencies    = []
tools_counts = []

for patient_id, prompt in test_cases:
    start = time.time()
    
    #simulating the tool decision step
    decision_prompt = f"""Decide which tools to call for this patient query.
Patient ID: {patient_id}
Patient medications: ["Metformin"]
User message: "{prompt}"
Available tools: check_drug_interactions, get_medication_cost,
lookup_nutrition, get_fda_alerts, search_pubmed
Reply ONLY with a JSON list. Example: ["lookup_nutrition"]"""

    resp = llm.invoke([HumanMessage(content=decision_prompt)])

    elapsed = time.time() - start
    latencies.append(elapsed)

    print(f"  {prompt[:40]:40s}: {elapsed:.2f}s")
avg_latency = np.mean(latencies)
print(f"\nAverage full agent latency: {avg_latency:.1f}s")
print(f"Average tools per query:    {np.mean(tools_counts):.1f}")
print(f"Min: {min(latencies):.1f}s  Max: {max(latencies):.1f}s")

METRIC_LATENCY = avg_latency
METRIC_AVG_TOOLS = np.mean(tools_counts)


METRIC 3: Full Agent Loop Latency
Measuring complete pipeline: context + tools + LLM response
--------------------------------------------------
  Is idli okay for breakfast?             : 0.31s
  How much could I save on my medications?: 0.12s
  What foods should I avoid?              : 0.17s
  My glucose was 140 today, is that okay? : 0.11s
  Tell me about the side effects of my med: 0.25s

Average full agent latency: 0.2s
Average tools per query:    nan
Min: 0.1s  Max: 0.3s


C:\Users\MAITHILI\anaconda3\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\MAITHILI\anaconda3\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [12]:
print("=" * 55)
print("CARECOMPANION AI — EVALUATION SUMMARY")
print("=" * 55)

metrics = {
    "Tool Call Success Rate":     f"{METRIC_TOOL_SUCCESS:.1f}%",
    "Memory Retrieval Accuracy":  f"{METRIC_MEMORY_ACCURACY:.1f}%",
    "Avg Annual Cost Savings":    f"${METRIC_AVG_COST_SAVINGS:,.0f}/patient",
    "Nutrition Assessment Accuracy": f"{METRIC_NUTRITION_ACCURACY:.1f}%",
    "FDA Alert Coverage":         f"{METRIC_FDA_COVERAGE:.1f}%",
    "Avg Reasoning Latency":      f"{METRIC_LATENCY:.2f}s",
}

print()
for name, value in metrics.items():
    print(f"  {name:40s}: {value}")

#saving to file
results_path = ROOT / "evaluation" / "metrics_report.json"
report = {
    "generated_at":             datetime.now().isoformat(),
    "patients_tested":          len(test_patients),
    "tool_success_rate_pct":    METRIC_TOOL_SUCCESS,
    "memory_accuracy_pct":      METRIC_MEMORY_ACCURACY,
    "avg_annual_cost_savings":  METRIC_AVG_COST_SAVINGS,
    "nutrition_accuracy_pct":   METRIC_NUTRITION_ACCURACY,
    "fda_coverage_pct":         METRIC_FDA_COVERAGE,
    "avg_latency_seconds":      METRIC_LATENCY,
}

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print()
print(f"Report saved: {results_path}")

CARECOMPANION AI — EVALUATION SUMMARY

  Tool Call Success Rate                  : 81.8%
  Memory Retrieval Accuracy               : 66.7%
  Avg Annual Cost Savings                 : $2,294/patient
  Nutrition Assessment Accuracy           : 70.0%
  FDA Alert Coverage                      : 100.0%
  Avg Reasoning Latency                   : 0.19s

Report saved: C:\Users\MAITHILI\Care_Companion\codes\carecompanion-ai\evaluation\metrics_report.json
